
# K-Nearest Neighbors (KNN) Classification using Salary Dataset

## Dataset
Uploaded Dataset: `salary_data.csv`

## Objective
Predict whether a salary is:
- High Salary
- Low Salary

based on:
- Years of Experience

The salary column is converted into binary classes using the median salary value.

## Features Included
- Automatic dataset loading
- Exploratory Data Analysis (EDA)
- Missing value handling
- Feature scaling
- Train/Test/Validation split
- KNN Classification
- Hyperparameter tuning
- Confusion Matrix
- ROC Curve
- Elbow Method
- 2D Decision Boundary Plot


In [ ]:

# =========================
# Import Libraries
# =========================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

# =========================
# Load Dataset
# =========================

df = pd.read_csv('/content/salary_data.csv')

print("First 5 Rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)


In [ ]:

# =========================
# Dataset Information
# =========================

print("\nDataset Info:\n")
print(df.info())

print("\nMissing Values:\n")
print(df.isnull().sum())

print("\nStatistical Summary:\n")
print(df.describe())

# =========================
# Exploratory Data Analysis
# =========================

# Salary Distribution

plt.figure(figsize=(7,5))

sns.histplot(df['Salary'], bins=10, kde=True)

plt.title("Salary Distribution")

plt.show()

# Years of Experience Distribution

plt.figure(figsize=(7,5))

sns.histplot(df['YearsExperience'], bins=10, kde=True)

plt.title("Years of Experience Distribution")

plt.show()

# Scatter Plot

plt.figure(figsize=(7,5))

sns.scatterplot(
    x='YearsExperience',
    y='Salary',
    data=df
)

plt.title("YearsExperience vs Salary")

plt.show()


In [ ]:

# =========================
# Data Preprocessing
# =========================

# Handle missing values

df.dropna(inplace=True)

# =========================
# Convert Salary into Classes
# =========================

median_salary = df['Salary'].median()

df['SalaryClass'] = np.where(
    df['Salary'] >= median_salary,
    1,
    0
)

print("\nMedian Salary:", median_salary)

print("\nClass Distribution:\n")

print(df['SalaryClass'].value_counts())

# =========================
# Features and Target
# =========================

X = df[['YearsExperience']]

y = df['SalaryClass']

# =========================
# Train / Validation / Test Split
# =========================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("\nTraining Size:", X_train.shape)

print("Validation Size:", X_val.shape)

print("Test Size:", X_test.shape)

# =========================
# Feature Scaling
# =========================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_val = scaler.transform(X_val)

X_test = scaler.transform(X_test)


In [ ]:

# =========================
# Elbow Method
# =========================

error_rate = []

for k in range(1, 16):

    knn = KNeighborsClassifier(n_neighbors=k)

    knn.fit(X_train, y_train)

    pred_k = knn.predict(X_test)

    error = np.mean(pred_k != y_test)

    error_rate.append(error)

# Plot

plt.figure(figsize=(8,5))

plt.plot(
    range(1,16),
    error_rate,
    marker='o'
)

plt.xlabel("K Value")

plt.ylabel("Error Rate")

plt.title("Error Rate vs K Value")

plt.show()


In [ ]:

# =========================
# Hyperparameter Tuning
# =========================

param_grid = {
    'n_neighbors': list(range(1, 11)),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

print("Best Parameters:\n")

print(grid.best_params_)

best_model = grid.best_estimator_

# =========================
# Validation Accuracy
# =========================

val_pred = best_model.predict(X_val)

print("\nValidation Accuracy:")

print(accuracy_score(y_val, val_pred))


In [ ]:

# =========================
# Final Prediction
# =========================

y_pred = best_model.predict(X_test)

y_prob = best_model.predict_proba(X_test)[:,1]

# =========================
# Evaluation Metrics
# =========================

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

auc = roc_auc_score(y_test, y_prob)

print("\n===== Evaluation Metrics =====")

print("Accuracy :", accuracy)

print("Precision:", precision)

print("Recall   :", recall)

print("F1 Score :", f1)

print("AUC Score:", auc)

print("\nClassification Report:\n")

print(classification_report(y_test, y_pred))

# =========================
# Confusion Matrix
# =========================

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("Confusion Matrix")

plt.show()

# =========================
# ROC Curve
# =========================

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(6,5))

plt.plot(fpr, tpr, label='KNN ROC Curve')

plt.plot([0,1], [0,1], linestyle='--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()


In [ ]:

# =========================
# Sample Predictions
# =========================

sample_df = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred[:10],
    'Probability': y_prob[:10]
})

print(sample_df)


In [ ]:

# =========================
# 2D Decision Boundary Plot
# =========================

x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1

xx = np.arange(x_min, x_max, 0.01)

xx_grid = xx.reshape(-1,1)

Z = best_model.predict(xx_grid)

plt.figure(figsize=(8,5))

plt.scatter(
    X_train[:,0],
    y_train,
    c=y_train,
    edgecolors='k'
)

plt.plot(xx, Z)

plt.xlabel("Scaled YearsExperience")

plt.ylabel("Salary Class")

plt.title("2D Decision Boundary Plot")

plt.show()



# Conclusion

This project implemented a complete KNN classification workflow using the uploaded Salary Dataset.

The workflow included:
- Data preprocessing
- Feature scaling
- Hyperparameter tuning
- Model evaluation
- Visualization techniques

The model performance was evaluated using:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
